In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!rm -rf /content/music-genre-classification
!cp -r "/content/drive/MyDrive/music-genre-classification" /content/music-genre-classification
%cd /content/music-genre-classification

!pip install -q -r requirements.txt

/content/music-genre-classification


In [4]:
import subprocess
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

downloads = [
    (
        "fma_metadata",
        "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip",
        data_dir / "fma_metadata.zip",
        data_dir / "fma_metadata",
    ),
    (
        "fma_small",
        "https://os.unil.cloud.switch.ch/fma/fma_small.zip",
        data_dir / "fma_small.zip",
        data_dir / "fma_small",
    ),
]

for name, url, archive_path, extracted_path in downloads:
    if extracted_path.exists():
        print(f"{name} ready:", extracted_path)
        continue

    if not archive_path.exists():
        print(f"Downloading {name}...")
        subprocess.run(["wget", "-c", "-O", str(archive_path), url], check=True)
    else:
        print(f"Using existing archive:", archive_path)

    print(f"Extracting {name}...")
    subprocess.run(["unzip", "-q", "-n", str(archive_path), "-d", str(data_dir)], check=True)

required = [Path("features/mel_specs.npz"), Path("features/mel_segments.npz")]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing cached features: " + ", ".join(missing))
print("Feature caches ready:", ", ".join(str(path) for path in required))

audio_root = Path("data/fma_small")
print("Raw audio ready for audio transfer learning:", audio_root)

Extracting fma_metadata...
Extracting fma_small...
Feature caches ready: features/mel_specs.npz, features/mel_segments.npz
Raw audio ready for audio transfer learning: data/fma_small


In [ ]:
from pathlib import Path

if Path("data/fma_small").exists():
    !python3 transfer_learning_ast.py
else:
    print("Skipping 5.3 AST because data/fma_small is missing.")


[Load AudioSet-pretrained AST]
Model: MIT/ast-finetuned-audioset-10-10-0.4593
preprocessor_config.json: 100% 297/297 [00:00<00:00, 1.35MB/s]
config.json: 100% 26.8k/26.8k [00:00<00:00, 67.5MB/s]
model.safetensors: 100% 346M/346M [00:02<00:00, 121MB/s] 
Loading weights: 100% 199/199 [00:00<00:00, 6865.39it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

[Extract AST embeddings]
Tracks: 7997  crop_duration=10.0s  segments=4x10.0s
AST batches:   6% 61/1000 [04:53<1:16:15,  4.87s/it][src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization fai

In [ ]:
import pandas as pd
from pathlib import Path

path = Path("results/5.3 AST/metrics.csv")
if not path.exists():
    raise FileNotFoundError("Missing results/5.3 AST/metrics.csv. Run transfer_learning_ast.py first.")
metrics = pd.read_csv(path)
metrics[metrics["split"] == "test"].sort_values("f1_macro", ascending=False)

,model,split,accuracy,f1_macro,best_epoch,best_val_f1,param_count,trainable_param_count,training_seconds,epochs_run
3,AST embeddings - MLP - Segment Averaging,test,0.688125,0.686572,63,0.692458,460424,460424,88.689201,63
1,AST embeddings - MLP,test,0.638750,0.630280,13,0.645802,460424,460424,3.910522,13


In [ ]:
import shutil
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/music-genre-classification")
drive_results = drive_root / "results"
drive_features = drive_root / "features"
drive_results.mkdir(parents=True, exist_ok=True)
drive_features.mkdir(parents=True, exist_ok=True)

src = Path("results/5.3 AST")
if src.exists():
    dst = drive_results / "5.3 AST"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print("Copied", src, "->", dst)

for filename in ["model_comparison.csv", "model_comparison.png"]:
    src = Path("results") / filename
    if src.exists():
        shutil.copy2(src, drive_results / filename)

embedding_cache = Path("features/ast_embeddings.npz")
if embedding_cache.exists():
    shutil.copy2(embedding_cache, drive_features / embedding_cache.name)


Copied results/5.3 AST -> /content/drive/MyDrive/music-genre-classification/results/5.3 AST


In [5]:
import pandas as pd
from pathlib import Path
import shutil

In [ ]:
!python3 transfer_learning_fine_tuning.py

drive_results = drive_root / "results"
drive_results.mkdir(parents=True, exist_ok=True)

src = Path("results/5.4 Fine Tuning")
dst = drive_results / "5.4 Fine Tuning"
if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print("Copied", src, "->", dst)

for filename in ["model_comparison.csv", "model_comparison.png"]:
    src = Path("results") / filename
    if src.exists():
        shutil.copy2(src, drive_results / filename)
        print("Copied", src, "->", drive_results / filename)


[5.4 Fine Tuning selection]
     experiment                                            model  f1_macro  accuracy
        5.3 AST         AST embeddings - MLP - Segment Averaging  0.692458  0.694531
5.2 PANNs-CNN14 PANNs-CNN14 embeddings - MLP - Segment Averaging  0.648849  0.652344
5.2 PANNs-CNN14                     PANNs-CNN14 embeddings - MLP  0.646411  0.646875
        5.3 AST                             AST embeddings - MLP  0.645802  0.651563
Selected: AST embeddings - MLP - Segment Averaging from 5.3 AST
preprocessor_config.json: 100% 297/297 [00:00<00:00, 1.74MB/s]
config.json: 100% 26.8k/26.8k [00:00<00:00, 58.8MB/s]
[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `527`.
model.safetensors: 100% 346M/346M [00:02<00:00, 120MB/s]
Loading weights: 100% 203/203 [00:00<00:00, 5225.50it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |              

In [ ]:
from google.colab import runtime
runtime.unassign()